# DermaXplain - Data Preprocessing Notebook

This notebook defines the **core preprocessing pipeline** for DermaXplain.

## Objectives

- load the inventory manifests produced by `01_data_inventory.ipynb`
- define reusable image and mask preprocessing helpers
- verify binary mask handling
- define resizing rules for images and masks
- define normalisation for CNN input
- visually verify resized image/mask alignment
- extract lesion geometry features from segmentation masks
- save enriched manifests for later modelling and XAI notebooks


Further experimental branches should go into a separate notebook such as:
`01_preprocessing_experiments.ipynb`


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)


In [ ]:
# Adjust if notebook location changes
ROOT = Path("..")

DATA_DIR = ROOT / "data"
MANIFEST_DIR = DATA_DIR / "preprocessed_manifests"
OUT_DIR = DATA_DIR / "preprocessed_manifests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ISIC_DIR = DATA_DIR / "isic2018"
ISIC_IMG_DIR = ISIC_DIR / "images"
ISIC_MASK_DIR = ISIC_DIR / "masks"

HAM_DIR = DATA_DIR / "ham10000"
HAM_IMG_DIR = HAM_DIR / "images"
HAM_MASK_DIR = HAM_DIR / "masks"

ISIC_MANIFEST = MANIFEST_DIR / "isic2018_inventory.csv"
HAM_MANIFEST = MANIFEST_DIR / "ham10000_inventory.csv"

assert ISIC_MANIFEST.exists(), f"Missing manifest: {ISIC_MANIFEST.resolve()}"
assert HAM_MANIFEST.exists(), f"Missing manifest: {HAM_MANIFEST.resolve()}"

print("Using manifests:")
print("-", ISIC_MANIFEST.resolve())
print("-", HAM_MANIFEST.resolve())


## 1. Load manifests

The inventory notebook should already have created one row per image with:
- path matching information
- size checks
- mask coverage
- border-touch diagnostics
- metadata linkage for HAM10000


In [ ]:

isic_df = pd.read_csv(ISIC_MANIFEST)
ham_df = pd.read_csv(HAM_MANIFEST)

if "lesion_id" not in ham_df.columns:
    raise ValueError(
        "HAM inventory does not contain lesion_id. "
        "Run the updated 01_data_inventory_lesion_id.ipynb first."
    )

if ham_df["lesion_id"].isna().any():
    raise ValueError("HAM inventory contains missing lesion_id values.")

print("ISIC shape:", isic_df.shape)
print("HAM shape :", ham_df.shape)
print("HAM unique images :", ham_df["stem"].nunique())
print("HAM unique lesions:", ham_df["lesion_id"].nunique())

display(isic_df.head())
display(ham_df[["stem", "lesion_id", "dx", "label_name"]].head())


In [ ]:

# Keep only rows with both image and mask present and the same dimensions.
# All metadata columns, including lesion_id, are retained.
isic_core = isic_df.loc[
    isic_df["image_exists"] & isic_df["mask_exists"] & isic_df["same_size"]
].copy()

ham_core = ham_df.loc[
    ham_df["image_exists"] & ham_df["mask_exists"] & ham_df["same_size"]
].copy()

assert "lesion_id" in ham_core.columns
assert ham_core["lesion_id"].notna().all()

print("ISIC usable rows:", len(isic_core))
print("HAM usable rows :", len(ham_core))
print("HAM usable unique lesions:", ham_core["lesion_id"].nunique())


## 2. Helper functions


In [ ]:
# 1- Load image and mask from disk
def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)


def load_mask_gray(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))

# 2- Convert grayscale mask to binary (0 and 1)
def to_mask_bin(mask_gray: np.ndarray) -> np.ndarray:
    return (mask_gray > 0).astype(np.uint8)

# 3- Overlay binary mask on RGB image for visualisation (red color with alpha blending)
def overlay_mask(img_rgb: np.ndarray, mask_bin: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    out = img_rgb.copy().astype(np.float32)
    color = np.zeros_like(out)
    color[..., 0] = 255  # red
    lesion = mask_bin.astype(bool)
    out[lesion] = (1 - alpha) * out[lesion] + alpha * color[lesion]
    return out.astype(np.uint8)

# 4- Image resizing

## Use bilinear interpolation for RGB images to maintain visual quality
def resize_image_rgb(img_rgb: np.ndarray, size=(224, 224)) -> np.ndarray:
    return np.array(Image.fromarray(img_rgb).resize(size, Image.Resampling.BILINEAR))

## Use nearest neighbor for masks to preserve binary nature
def resize_mask_bin(mask_bin: np.ndarray, size=(224, 224)) -> np.ndarray:
    mask_img = Image.fromarray((mask_bin * 255).astype(np.uint8))
    mask_resized = mask_img.resize(size, Image.Resampling.NEAREST)
    return (np.array(mask_resized) > 0).astype(np.uint8)

## 5- Normalisation for pretrained models, e.g. ResNet, EfficientNet, etc.
## mean and std from ImageNet, as commonly used for pretrained models
def normalize_imagenet(img_rgb: np.ndarray) -> np.ndarray:
    x = img_rgb.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return (x - mean) / std


## 3. Binary mask handling

Masks may be stored as grayscale PNG images. For geometry and evaluation work we convert them to a clean binary representation.


In [ ]:
def sample_triplet(df: pd.DataFrame, img_dir: Path, mask_dir: Path, n: int = 4, random_state: int = 42):
    sample = df.sample(min(n, len(df)), random_state=random_state)
    fig, axes = plt.subplots(len(sample), 3, figsize=(12, 4 * len(sample)))

    if len(sample) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(sample.iterrows()):
        stem = row["stem"]
        img = load_rgb(img_dir / f"{stem}.jpg")
        mask_gray = load_mask_gray(mask_dir / f"{stem}.png")
        mask_bin = to_mask_bin(mask_gray)
        overlay = overlay_mask(img, mask_bin)

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{stem} - image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask_gray, cmap="gray")
        axes[i, 1].set_title("original mask")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f"binary overlay (coverage={mask_bin.mean():.3f})")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Visual check on HAM10000
sample_triplet(ham_core, HAM_IMG_DIR, HAM_MASK_DIR, n=4, random_state=7)


## 4. Resizing strategy

For the baseline CNN we use:
- **bilinear interpolation** for RGB images
- **nearest-neighbor interpolation** for binary masks

Nearest-neighbor must be used for masks to avoid creating artificial gray values on boundaries.

Note: we prefer to resize images and masks to 224x224 for compatibility with common pretrained models and to reduce memory usage during training.

All geometric features later in the pipeline (e.g. feature extraction, classification, lesion diameter) will be based on these resized versions. 
Therefore they represent relative rather than absolute physical measurements, but this is acceptable for our classification task.
Resizing changes absolute geometry, but not relative structure; diameter becomes relative (pixel-based); pseudo-concepts still work correctly.
The only caveat is aspect ratio distortion. 

However using 224x224 ratio for all model input, masks, saliency, pseudo-concepts, metrics will not cause any issues, as all will be resized to the same dimensions.
 
For visualisation, we can still use the original aspect ratio images and masks, as they are only for qualitative assessment and not used in the model training or evaluation.


In [ ]:
def show_resized_examples(df: pd.DataFrame, img_dir: Path, mask_dir: Path, n: int = 3, size=(224, 224), random_state: int = 42):
    sample = df.sample(min(n, len(df)), random_state=random_state)
    fig, axes = plt.subplots(len(sample), 4, figsize=(15, 4 * len(sample)))

    if len(sample) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(sample.iterrows()):
        stem = row["stem"]
        img = load_rgb(img_dir / f"{stem}.jpg")
        mask = to_mask_bin(load_mask_gray(mask_dir / f"{stem}.png"))

        img_resized = resize_image_rgb(img, size=size)
        mask_resized = resize_mask_bin(mask, size=size)
        overlay_resized = overlay_mask(img_resized, mask_resized)

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{stem} - original image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title("original mask")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(img_resized)
        axes[i, 2].set_title(f"resized image {size}")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(overlay_resized)
        axes[i, 3].set_title("resized overlay")
        axes[i, 3].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
show_resized_examples(ham_core, HAM_IMG_DIR, HAM_MASK_DIR, n=3, size=(224, 224), random_state=11)


## 5. Normalisation strategy

For the ResNet-50 baseline we use standard **ImageNet normalisation** after scaling RGB values to `[0, 1]`.

This notebook defines the logic and performs a quick sanity check. The actual tensor conversion can be handled later in the training notebook.


In [ ]:
# Quick normalisation sanity check on one image
example_stem = ham_core.iloc[0]["stem"]
example_img = load_rgb(HAM_IMG_DIR / f"{example_stem}.jpg")
example_img_224 = resize_image_rgb(example_img, size=(224, 224))
example_norm = normalize_imagenet(example_img_224)

print("Original dtype:", example_img_224.dtype)
print("Normalised dtype:", example_norm.dtype)
print("Normalised shape:", example_norm.shape)
print("Channel-wise min:", example_norm.min(axis=(0, 1)))
print("Channel-wise max:", example_norm.max(axis=(0, 1)))
print("Channel-wise mean:", example_norm.mean(axis=(0, 1)))


## 6. Lesion geometry extraction

We derive a small set of lesion-level geometric descriptors directly from the binary mask:

- `mask_pixels`
- `mask_coverage`
- `equivalent_diameter_px`
- `centroid_x`, `centroid_y`
- bounding box coordinates and dimensions

These features are useful later for:
- analysing lesion size effects on XAI metrics
- generating pseudo-concept regions
- cropping or centering experiments


In [ ]:
def extract_geometry_from_mask(mask_bin: np.ndarray) -> dict:
    ys, xs = np.where(mask_bin > 0)

    area = int(mask_bin.sum())

    if area == 0:
        return {
            "mask_pixels": 0,
            "mask_coverage": 0.0,
            "equivalent_diameter_px": 0.0,
            "centroid_x": np.nan,
            "centroid_y": np.nan,
            "bbox_xmin": np.nan,
            "bbox_ymin": np.nan,
            "bbox_xmax": np.nan,
            "bbox_ymax": np.nan,
            "bbox_width": np.nan,
            "bbox_height": np.nan,
        }

    y_min, y_max = int(ys.min()), int(ys.max())
    x_min, x_max = int(xs.min()), int(xs.max())

    h, w = mask_bin.shape
    coverage = area / float(h * w)
    eq_diameter = float(np.sqrt(4.0 * area / np.pi))
    cx = float(xs.mean())
    cy = float(ys.mean())

    return {
        "mask_pixels": area,
        "mask_coverage": coverage,
        "equivalent_diameter_px": eq_diameter,
        "centroid_x": cx,
        "centroid_y": cy,
        "bbox_xmin": x_min,
        "bbox_ymin": y_min,
        "bbox_xmax": x_max,
        "bbox_ymax": y_max,
        "bbox_width": x_max - x_min + 1,
        "bbox_height": y_max - y_min + 1,
    }


In [ ]:
def enrich_with_geometry(df: pd.DataFrame, mask_dir: Path) -> pd.DataFrame:
    records = []

    for _, row in df.iterrows():
        stem = row["stem"]
        mask_path = mask_dir / f"{stem}.png"
        mask_bin = to_mask_bin(load_mask_gray(mask_path))
        geom = extract_geometry_from_mask(mask_bin)
        geom["stem"] = stem
        records.append(geom)

    geom_df = pd.DataFrame(records)
    return df.merge(geom_df, on="stem", how="left")


In [ ]:
## Note this takes a few minutes to run as it processes all images and masks to extract geometry features.
isic_pre = enrich_with_geometry(isic_core, ISIC_MASK_DIR)
ham_pre = enrich_with_geometry(ham_core, HAM_MASK_DIR)

display(isic_pre.head())
display(ham_pre.head())


In [ ]:
print("ISIC geometry summary:")
display(isic_pre[[
    "mask_pixels_x", "mask_coverage_x", "equivalent_diameter_px",
    "bbox_width", "bbox_height"
]].describe())

print("HAM geometry summary:")
display(ham_pre[[
    "mask_pixels_x", "mask_coverage_x", "equivalent_diameter_px",
    "bbox_width", "bbox_height"
]].describe())


In [ ]:
# ============================================================
# Combined lesion-size distribution: HAM10000 + ISIC2018

import pandas as pd

SIZE_ORDER = [
    "normal",
    "small",
    "tiny",
    "very_tiny",
]

SIZE_LABELS = {
    "normal": "Normal",
    "small": "Small",
    "tiny": "Tiny",
    "very_tiny": "Very Tiny",
}

combined = pd.concat(
    [
        ham_pre.assign(dataset="HAM10000"),
        isic_pre.assign(dataset="ISIC2018"),
    ],
    ignore_index=True
)

combined["mask_size_class"] = (
    combined["mask_size_class"]
    .astype(str)
    .str.strip()
    .str.lower()
)

total_n = len(combined)

scale_table = (
    combined["mask_size_class"]
    .value_counts()
    .reindex(SIZE_ORDER, fill_value=0)
    .rename_axis("mask_size_class")
    .reset_index(name="Count")
)

scale_table["Scale Tier"] = (
    scale_table["mask_size_class"]
    .map(SIZE_LABELS)
)

scale_table["Dataset %"] = (
    100 * scale_table["Count"] / total_n
).round(1)

scale_table = scale_table[
    ["Scale Tier", "Count", "Dataset %"]
]

display(scale_table)

In [ ]:
# ============================================================
# Combined border-touch prevalence: HAM10000 + ISIC2018


border_cols = [
    "touch_top",
    "touch_bottom",
    "touch_left",
    "touch_right",
    "touches_any_border",
]

# Ensure Boolean values
for col in border_cols:
    if combined[col].dtype != bool:
        combined[col] = (
            combined[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            })
        )

border_table = pd.DataFrame({
    "Border Condition": [
        "Top",
        "Bottom",
        "Left",
        "Right",
        "Any Border",
    ],
    "Count": [
        combined["touch_top"].sum(),
        combined["touch_bottom"].sum(),
        combined["touch_left"].sum(),
        combined["touch_right"].sum(),
        combined["touches_any_border"].sum(),
    ],
})

border_table["Dataset %"] = (
    100 * border_table["Count"] / len(combined)
).round(1)

display(border_table)

## 7. Visual geometry sanity check


In [ ]:
def show_geometry_examples(df: pd.DataFrame, img_dir: Path, mask_dir: Path, n: int = 4, random_state: int = 42):
    sample = df.sample(min(n, len(df)), random_state=random_state)
    fig, axes = plt.subplots(len(sample), 2, figsize=(11, 4 * len(sample)))

    if len(sample) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, (_, row) in enumerate(sample.iterrows()):
        stem = row["stem"]
        img = load_rgb(img_dir / f"{stem}.jpg")
        mask = to_mask_bin(load_mask_gray(mask_dir / f"{stem}.png"))
        overlay = overlay_mask(img, mask)

        x0, y0 = int(row["bbox_xmin"]), int(row["bbox_ymin"])
        bw, bh = int(row["bbox_width"]), int(row["bbox_height"])

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{stem} - original")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(overlay)
        rect = plt.Rectangle((x0, y0), bw, bh, fill=False, edgecolor="yellow", linewidth=2)
        axes[i, 1].add_patch(rect)
        axes[i, 1].scatter(row["centroid_x"], row["centroid_y"], s=25, c="cyan")
        axes[i, 1].set_title(
            f"overlay | eq_diam={row['equivalent_diameter_px']:.1f}px | coverage={row['mask_coverage_x']:.3f}"
        )
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
show_geometry_examples(ham_pre, HAM_IMG_DIR, HAM_MASK_DIR, n=4, random_state=21)


## 8. Save enriched preprocessing manifests

We export enriched manifests that can be reused later by:
- the training notebook
- the XAI notebook
- the pseudo-concept generation notebook


In [ ]:

isic_out = OUT_DIR / "isic2018_preprocessed.csv"
ham_out = OUT_DIR / "ham10000_preprocessed.csv"

# lesion_id is carried through enrich_with_geometry because the geometry
# table is merged back onto the complete source manifest by stem.
assert "lesion_id" in ham_pre.columns
assert ham_pre["lesion_id"].notna().all()

isic_pre.to_csv(isic_out, index=False)
ham_pre.to_csv(ham_out, index=False)

print("Saved:", isic_out.resolve())
print("Saved:", ham_out.resolve())
print("ISIC rows:", len(isic_pre))
print("HAM rows :", len(ham_pre))
print("HAM unique lesions:", ham_pre["lesion_id"].nunique())


## Summary

This notebook completed the **core preprocessing definition** for DermaXplain:

- loaded and filtered inventory manifests
- confirmed binary mask conversion
- defined resizing rules for images and masks
- defined ImageNet normalization for CNN input
- extracted lesion geometry descriptors in pixel space
- exported enriched preprocessing manifests

### Next notebooks

- `01_preprocessing_experiments.ipynb` for optional branches such as cropping, hair removal, and color normalization
- `02_model_baseline_resnet50.ipynb` for CNN training
